#### Code Generation along with Gradio UI

In [17]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
import gradio as gr

import io
import sys

In [2]:
# load keys

load_dotenv(override=True)

gemini_base_url = os.getenv('GEMINI_BASE_URL')
gemini_api_key = os.getenv('GEMINI_API_KEY')

groq_base_url = os.getenv('GROQ_BASE_URL')
groq_api_key = os.getenv('GROQ_API_KEY')

if gemini_api_key:
    print(f'Gemini API Key found and starts with {gemini_api_key[0:3]}')
else:
    print('Gemini API key not found')

if groq_api_key:
    print(f'Groq API Key found and starts with {groq_api_key[0:3]}')
else:
    print('Groq API Key not found')

Gemini API Key found and starts with AQ.
Groq API Key found and starts with gsk


In [3]:
# clients

gemini = OpenAI(base_url = gemini_base_url, api_key = gemini_api_key)
groq = OpenAI(base_url = groq_base_url, api_key = groq_api_key)

In [4]:
# models

gemini_model = 'gemini-3.6-flash'
gpt_model_20b = 'openai/gpt-oss-20b'
gpt_model_120b = 'openai/gpt-oss-120b'
groq_model = 'groq/compound-mini'

In [5]:
# models and clients

models = ['gemini-3.6-flash', 'openai/gpt-oss-20b', 'openai/gpt-oss-120b', 'groq/compound-mini']
clients = {"gemini-3.6-flash":gemini, "openai/gpt-oss-20b":groq, "openai/gpt-oss-120b":groq, "groq/compound-mini":groq}

In [9]:
# to retrieve system info

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:26:46 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T8103',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M1',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [10]:
# compile_command and run_command from day_3 of 004_week

compile_command = ["clang++", "-O3", "-mcpu=native", "main.cpp", "-o", "main"]
run_command = ["./main"]

#### Task

In [12]:
# system_prompt and user_prompt

system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt(python_code):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python_code}
```
"""

In [13]:
# complete prompt messages

def messages(python_code):
    return [
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_prompt(python_code)}
    ]

In [14]:
# write Cpp to main.cpp

def write_cpp(cpp):
    with open('main.cpp', 'w') as f:
        f.write(cpp)

In [15]:
# to convert from python to Cpp

def port(model, python_code):
    client=clients[model]
    response = client.chat.completions.create(
        model = model,
        messages = messages(python_code)
    )
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_cpp(reply)
    return reply

In [16]:
# Python code to determine pi-value

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [25]:
# to run python code from string

def run_python_code(code):
    globals_dict = {"__builtins__":__builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f'Error: {e}'
    finally:
        sys.stdout = old_stdout
    return output

In [27]:
# run the Python version of code

print(run_python_code(pi))

Result: 3.141592656089
Execution Time: 21.634684 seconds



In [20]:
# to compile the generated Cpp code

def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    except subprocess.CalledProcessError as e:
        print(f'An Error occured:\n{e.stderr}')


In [23]:
# gradio ui

with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label='Python Code', lines=28, value=pi)
        cpp = gr.Textbox(label='Cpp Code', lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label='Select model', value=models[0])
        convert = gr.Button("Convert the Code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


In [29]:
# to compile the Cpp version of code

compile_and_run()

Result: 3.141592656091
Execution Time: 0.026325 seconds

Result: 3.141592656091
Execution Time: 0.025485 seconds

Result: 3.141592656091
Execution Time: 0.027189 seconds

